In [35]:
# чтобы правки в src/features.py подхватывались автоматически, без перезапуска kernel.
%load_ext autoreload
%autoreload 2

# импорты основных библиотек
import numpy as np
import pandas as pd

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [36]:
# импорт основной таблицы
df = pd.read_csv('../data/processed/transactions_clean.csv')

In [37]:
# меняю тип данных дат с object на datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 417503 entries, 0 to 417502
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      417503 non-null  object        
 1   StockCode    417503 non-null  object        
 2   Description  417503 non-null  object        
 3   Quantity     417503 non-null  int64         
 4   InvoiceDate  417503 non-null  datetime64[ns]
 5   Price        417503 non-null  float64       
 6   Customer ID  417503 non-null  float64       
 7   Country      417503 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 25.5+ MB


In [38]:
order_dates = (
    df.groupby(['Customer ID', 'Invoice'])['InvoiceDate']
    .min()
    .reset_index()
    .sort_values(['Customer ID', 'InvoiceDate'])
)

# начинаем считать разницу между покупками клиентов и переводим даты в конкретные дни
# Благодаря groupby первая покупка клиента обозначается NaT благодаря чему не происходит
# 
order_dates.groupby('Customer ID')['InvoiceDate'].diff().dt.days

0          NaN
1          0.0
2          0.0
3          3.0
4          0.0
         ...  
23584      NaN
23580    166.0
23581    127.0
23582      0.0
23583     61.0
Name: InvoiceDate, Length: 23585, dtype: float64

In [39]:
# начинаем искать точку через какое количество времени считать клиента ушедшим

# сворачиваем транзакции до уровня заказа, одна строка = один Invoice (min даты).
# затем сортируем по клиенту и дате, diff ниже требует хронологического порядка.
order_dates = (
    df.groupby(['Customer ID', 'Invoice'])['InvoiceDate']
    .min()
    .reset_index()
    .sort_values(['Customer ID', 'InvoiceDate'])
)

# Считаем разницу между соседними покупками клиента и переводим её в дни.
# Благодаря groupby первая покупка каждого клиента помечается NaT за счёт этого
# не происходит вычитания через границу клиентов (дата одного минус дата другого).
# .dt.days переводит timedelta в число дней (NaT становится NaN).
days_since_prev_purch = order_dates.groupby('Customer ID')['InvoiceDate'].diff().dt.days

# добавляем столбец days_since_prev_purch в order_dates и убираем строки с NaN
order_dates['days_since_prev_purch'] = days_since_prev_purch
order_dates = order_dates.dropna()
display(order_dates)

# считаем квантили для определения оптимальной точки
print(order_dates['days_since_prev_purch'].quantile([0.9,0.95]))

,Customer ID,Invoice,InvoiceDate,days_since_prev_purch
1,12346.0,491742,2009-12-14 11:00:00,0.0
2,12346.0,491744,2009-12-14 11:02:00,0.0
3,12346.0,492718,2009-12-18 10:47:00,3.0
4,12346.0,492722,2009-12-18 10:55:00,0.0
5,12346.0,493410,2010-01-04 09:24:00,16.0
...,...,...,...,...
23578,18286.0,519785,2010-08-20 11:57:00,56.0
23580,18287.0,508581,2010-05-17 11:55:00,166.0
23581,18287.0,523289,2010-09-21 12:17:00,127.0
23582,18287.0,523290,2010-09-21 12:19:00,0.0


0.90     86.0
0.95    130.0
Name: days_since_prev_purch, dtype: float64


### Вывод: обоснование порога оттока

Посчитали интервалы между соседними заказами по всем клиентам:

| квантиль | дней |
|---|---|
| 90% | 86 |
| 95% | 130 |

**Интерпретация:** 90% всех пауз между покупками укладываются в 86 дней.
Значит молчание дольше ~86 дней попадает в верхние 10% — это уже
нетипичное поведение, а не «клиент просто ещё не собрался за покупкой».

**Решение:** берём порог **86 дней** (90-й перцентиль), а не 130.
Обоснование бизнесом: для интернет-магазина дороже **упустить** уходящего
клиента, чем зря побеспокоить лояльного акцией. Поэтому выбираем более
строгий порог — реагировать лучше раньше.

**Как используется дальше:** это число задаёт длину окна оттока.
Окно `after` берём равным **90 дням** (≈86) — то есть точка разреза
T = последняя дата данных − 90 дней. Клиент, не купивший за это окно,
молчит дольше 90% нормальных пауз → считаем его ушедшим.

Ключевое: порог взят **из данных**, а не назначен произвольно.


### Что дальше

Порог есть — теперь надо превратить его в **целевую переменную** (`churn`),
которой в датасете нет: готовой метки «ушёл / остался» здесь не существует,
её нужно сконструировать самим.

Дальше показываю два подхода:

1. **Наивный** — поставить метку напрямую: `churn = Recency > 86`.
   Разберу, почему так делать нельзя (спойлер: утечка целевой переменной).
2. **Правильный — временной сплит.** Режем период датой T:
   признаки считаем по данным **до T**, факт оттока — по покупкам **после T**.
   Так метка и признаки берутся из разных периодов, и модель учится
   предсказывать будущее, а не списывать ответ.

Итог этапа: таблица «признаки + целевая переменная» на каждого клиента,
готовая для обучения моделей на этапе 5.


### ❌ Наивный подход (демонстрация ошибки)

Первая, интуитивная мысль — определить отток напрямую из Recency:
`churn = Recency > 86`. Ниже показываю этот вариант **специально**, чтобы
разобрать, почему он не работает.

В финальном пайплайне он не используется: пишет в отдельную переменную
`rfm`, которая нигде дальше не участвует (настоящая таблица —
`rfm_before` ниже).

In [40]:
rfm = pd.read_csv('../data/processed/rfm_with_clusters.csv')

In [41]:
rfm['churn'] = (rfm['Recency'] > 86).astype(int)
print(rfm['churn'].value_counts())
rfm

churn
0    2391
1    1390
Name: count, dtype: int64


,Customer ID,Recency,Frequency,Monetary,Cluster,Cluster_name,churn
0,12347.0,3,2,1323.32,2,Обычные,0
1,12348.0,74,1,222.16,0,Новые,0
2,12349.0,43,4,2646.99,1,Лояльные,0
3,12351.0,11,1,300.93,0,Новые,0
4,12352.0,11,2,343.80,0,Новые,0
...,...,...,...,...,...,...,...
3776,18283.0,18,6,641.77,2,Обычные,0
3777,18284.0,65,2,436.68,0,Новые,0
3778,18285.0,296,1,427.00,3,Группа риска,1
3779,18286.0,112,3,1188.43,2,Обычные,1


### Почему так нельзя: утечка целевой переменной

Здесь `churn` посчитан напрямую из `Recency`, а `Recency` затем пойдёт
в признаки модели. Целевая переменная становится **функцией признака** —
модель просто выучит правило `Recency > 86`, покажет идеальные метрики и
окажется бесполезной на реальных данных. Это **утечка целевой переменной**
(target leakage): ответ «просачивается» в признаки.

Проверить легко: при такой метке классы 0 и 1 идеально разделяются по
`Recency` на границе 86 — без единого пересечения. В честных данных такой
идеальной границы по одному признаку не бывает, это красный флаг.

**Правильное решение — временной сплит.** Режем период датой T: признаки
считаем по данным **до T**, а факт оттока — по покупкам **после T**. Тогда
метка и признаки берутся из разных периодов, и `Recency@T` снова
становится честным информативным признаком (коррелирует с оттоком, но не
определяет его механически). Реализация — ниже. ⬇️

In [42]:
print(f"Минимальная дата: {df['InvoiceDate'].min()}")
print(f"Максимальная дата: {df['InvoiceDate'].max()}")

T = df['InvoiceDate'].max() - pd.Timedelta(days=90)
T

Минимальная дата: 2009-12-01 07:45:00
Максимальная дата: 2010-12-09 20:01:00


Timestamp('2010-09-10 20:01:00')

In [43]:
before = df[df['InvoiceDate'] < T]
after = df[df['InvoiceDate'] >= T]

### Фильтрация выбросов

На этапе 3 из выборки были удалены клиенты с экстремальными Frequency
и Monetary (вероятно оптовики) — метод IQR, правило Тьюки.

Здесь применяем **те же границы**. Причина не только в согласованности
скоупа: `scaler` и `KMeans` обучались на данных без выбросов, и подача
им значений в десятки раз шире обучающего диапазона превращает
`predict` в экстраполяцию — все выбросы механически падают в ближайший
центроид и искажают признак `Cluster`.


In [44]:
import sys
sys.path.append('../src')     
from features import compute_rfm


rfm_before = compute_rfm(before,T)

import joblib

# Применяем границы выбросов, зафиксированные на этапе 3.
# НЕ пересчитываем их заново — это часть обученного препроцессинга,
# по той же логике, по которой ниже используем scaler.transform, а не fit_transform.
bounds = joblib.load('../models/outlier_bounds.pkl')

rfm_before = rfm_before[
    (rfm_before['Monetary']  < bounds['upper_monetary']) &
    (rfm_before['Frequency'] < bounds['upper_frequency'])
].copy()

print(f"После фильтрации выбросов: {len(rfm_before)} клиентов")


После фильтрации выбросов: 3072 клиентов


In [45]:

id_after = after['Customer ID'].unique()
churnn = (~rfm_before.index.isin(id_after)).astype(int)
rfm_before['churn'] = churnn


In [46]:
rfm_before

,Recency,Frequency,Monetary,churn
Customer ID,,,,
12349.0,115,3,1244.37,0
12355.0,112,1,488.21,1
12358.0,95,2,1697.93,0
12359.0,80,7,1918.03,0
12360.0,108,4,740.04,0
...,...,...,...,...
18281.0,122,1,120.32,1
18283.0,22,4,446.42,0
18285.0,205,1,427.00,1


In [47]:
rfm_before['churn'].value_counts()

churn
0    1716
1    1356
Name: count, dtype: int64

In [48]:
import joblib

# 1. Загружаем обученные на этапе 3 объекты (возвращаются готовыми, с выученными параметрами)
scaler = joblib.load('../models/scaler.pkl')
kmeans = joblib.load('../models/kmeans_model.pkl')

# 2. Масштабируем признаки rfm_before — ВНИМАНИЕ: transform, НЕ fit_transform
#    (порядок колонок строго как при обучении: Recency, Frequency, Monetary)
scaled = scaler.transform(rfm_before[['Recency', 'Frequency', 'Monetary']])

# 3. Относим каждого клиента к ближайшему готовому центру кластера (predict, не fit)
rfm_before['Cluster'] = kmeans.predict(scaled)

# 4. Проверка глазами: сколько клиентов в каждом кластере
rfm_before['Cluster'].value_counts()


Cluster
0    1386
2     699
3     681
1     306
Name: count, dtype: int64

In [49]:
rfm_before

,Recency,Frequency,Monetary,churn,Cluster
Customer ID,,,,,
12349.0,115,3,1244.37,0,2
12355.0,112,1,488.21,1,0
12358.0,95,2,1697.93,0,2
12359.0,80,7,1918.03,0,1
12360.0,108,4,740.04,0,2
...,...,...,...,...,...
18281.0,122,1,120.32,1,0
18283.0,22,4,446.42,0,0
18285.0,205,1,427.00,1,3


In [ ]:
# Добавляем читаемые имена кластеров — понадобятся для интерпретации (SHAP, этап 7)
# и Streamlit-демо (этап 8). Маппинг тот же, что на этапе 3: номера кластеров
# совпадают, т.к. используется та же сохранённая модель kmeans_model.pkl.
cluster_names = {0: 'Новые', 1: 'Лояльные', 2: 'Обычные', 3: 'Группа риска'}
rfm_before['Cluster_name'] = rfm_before['Cluster'].map(cluster_names)

# Проверка, что имена соответствуют профилю: средние RFM по кластерам
# (Лояльные — низкий Recency/высокие Freq,Monetary; Группа риска — высокий Recency)
rfm_before.groupby('Cluster_name')[['Recency', 'Frequency', 'Monetary']].mean().round(1)

In [50]:
rfm_before.to_csv('../data/processed/churn_dataset.csv')
